In [2]:
import requests
import pandas as pd
import re
import time
from datetime import datetime, timedelta
from bs4 import BeautifulSoup
import warnings
warnings.filterwarnings('ignore')

In [3]:
class MeroLaganiHistoricalScraper:
    """
    MeroLagani Historical Data Scraper (1 Year)
    """

    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.9',
            'Connection': 'keep-alive',
        })
        self.all_data = []

    def clean_number(self, text):
        """Handle numbers with K/M/B suffixes"""
        if not text or text in ['', '-', 'N/A', 'null', None, '--', 'n/a']:
            return None
        try:
            text_str = str(text).strip().upper()
            multiplier = 1
            if text_str.endswith('K'):
                multiplier = 1000
                text_str = text_str[:-1]
            elif text_str.endswith('M'):
                multiplier = 1000000
                text_str = text_str[:-1]
            elif text_str.endswith('B'):
                multiplier = 1000000000
                text_str = text_str[:-1]
            cleaned = re.sub(r'[^\d.-]', '', text_str)
            if cleaned and cleaned not in ['-', '.', '']:
                return float(cleaned) * multiplier
        except:
            pass
        return None

    def get_trading_dates(self, start_date, end_date):
        """Generate list of trading days (skip Fri & Sat in Nepal + major holidays)"""
        dates = []
        current = start_date
        holidays = ['2024-01-01', '2024-04-13', '2024-10-24', '2024-11-12']
        while current <= end_date:
            if current.weekday() not in [4, 5]:  # Fri, Sat
                if current.strftime('%Y-%m-%d') not in holidays:
                    dates.append(current)
            current += timedelta(days=1)
        return dates

    def scrape_merolagani_historical(self, date):
        """Scrape MeroLagani data for a given date"""
        date_str = date.strftime('%Y-%m-%d')
        urls = [
            f"https://merolagani.com/LatestMarket.aspx?date={date_str}",
            f"https://www.merolagani.com/LatestMarket.aspx?date={date_str}"
        ]

        for url in urls:
            try:
                response = self.session.get(url, timeout=20)
                if response.status_code == 200:
                    soup = BeautifulSoup(response.content, 'html.parser')
                    tables = soup.find_all('table')
                    for table in tables:
                        rows = table.find_all('tr')
                        if len(rows) > 5:
                            stocks = []
                            header_found = False
                            for row in rows:
                                cells = [c.get_text(strip=True) for c in row.find_all(['td', 'th'])]
                                if not cells:
                                    continue
                                if any("symbol" in c.lower() for c in cells):
                                    header_found = True
                                    continue
                                if header_found and len(cells) >= 7:
                                    symbol = None
                                    if re.match(r'^[A-Z]{3,10}$', cells[0].strip()):
                                        symbol = cells[0].strip()
                                    if symbol:
                                        stock = {
                                            'date': date_str,
                                            'symbol': symbol,
                                            'close': self.clean_number(cells[1]),
                                            'change_percent': self.clean_number(cells[2]),
                                            'open': self.clean_number(cells[3]),
                                            'high': self.clean_number(cells[4]),
                                            'low': self.clean_number(cells[5]),
                                            'volume': self.clean_number(cells[6]),
                                            'source': 'merolagani_historical',
                                            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                                        }
                                        if stock['close'] is not None:
                                            stocks.append(stock)
                            if stocks:
                                return stocks
            except:
                continue
        return None

    def scrape_year_data(self, start_date=None, end_date=None, delay_seconds=2):
        """Fetch 1 year of data"""
        if not end_date:
            end_date = datetime.now()
        if not start_date:
            start_date = end_date - timedelta(days=365)

        trading_dates = self.get_trading_dates(start_date, end_date)
        for i, date in enumerate(trading_dates):
            print(f"[{i+1}/{len(trading_dates)}] {date.strftime('%Y-%m-%d')}")
            day_data = self.scrape_merolagani_historical(date)
            if day_data:
                self.all_data.extend(day_data)
            time.sleep(delay_seconds)
        return self.get_dataframe()

    def get_dataframe(self):
        """Return cleaned DataFrame with standardized schema"""
        if not self.all_data:
            return pd.DataFrame()
        df = pd.DataFrame(self.all_data)
        df['date'] = pd.to_datetime(df['date'], errors="coerce")
        df = df.drop_duplicates(subset=['symbol', 'date'])
        df = df.sort_values(['date', 'symbol']).reset_index(drop=True)

        # Ensure expected schema
        expected = ["date", "symbol", "open", "high", "low", "close",
                    "volume", "change_percent", "source", "timestamp"]
        df = df[[c for c in expected if c in df.columns]]
        return df

    def save_data(self, df, filename=None):
        """Save standardized historical data"""
        if df.empty:
            print("No data to save")
            return None
        if not filename:
            start_date = df['date'].min().strftime('%Y%m%d')
            end_date = df['date'].max().strftime('%Y%m%d')
            filename = f"MeroLagani_Historical_{start_date}_to_{end_date}.csv"
        df.to_csv(filename, index=False)
        print(f"Data saved to {filename}")
        return filename

In [4]:
# ------------------ USAGE ------------------
if __name__ == "__main__":
    scraper = MeroLaganiHistoricalScraper()
    
    # Define exact date range
    start_date = datetime(2024, 8, 25)
    end_date   = datetime(2025, 8, 25)

    # Fetch with custom range
    df = scraper.scrape_year_data(start_date=start_date, end_date=end_date, delay_seconds=3)
    
    if not df.empty:
        print(f"Collected {len(df)} records from {start_date.date()} to {end_date.date()}")
        scraper.save_data(df, filename="D:/TRADING_SYSTEM/backend/core/data/NEPSE_20240825_to_20250825.csv")
        print(df.head())
    else:
        print("Failed to fetch historical data")

[1/260] 2024-08-25
[2/260] 2024-08-26
[3/260] 2024-08-27
[4/260] 2024-08-28
[5/260] 2024-08-29
[6/260] 2024-09-01
[7/260] 2024-09-02
[8/260] 2024-09-03
[9/260] 2024-09-04
[10/260] 2024-09-05
[11/260] 2024-09-08
[12/260] 2024-09-09
[13/260] 2024-09-10
[14/260] 2024-09-11
[15/260] 2024-09-12
[16/260] 2024-09-15
[17/260] 2024-09-16
[18/260] 2024-09-17
[19/260] 2024-09-18
[20/260] 2024-09-19
[21/260] 2024-09-22
[22/260] 2024-09-23
[23/260] 2024-09-24
[24/260] 2024-09-25
[25/260] 2024-09-26
[26/260] 2024-09-29
[27/260] 2024-09-30
[28/260] 2024-10-01
[29/260] 2024-10-02
[30/260] 2024-10-03
[31/260] 2024-10-06
[32/260] 2024-10-07
[33/260] 2024-10-08
[34/260] 2024-10-09
[35/260] 2024-10-10
[36/260] 2024-10-13
[37/260] 2024-10-14
[38/260] 2024-10-15
[39/260] 2024-10-16
[40/260] 2024-10-17
[41/260] 2024-10-20
[42/260] 2024-10-21
[43/260] 2024-10-22
[44/260] 2024-10-23
[45/260] 2024-10-27
[46/260] 2024-10-28
[47/260] 2024-10-29
[48/260] 2024-10-30
[49/260] 2024-10-31
[50/260] 2024-11-03
[51/260] 